# 02. Raw baseline on LFW (E1)

Cosine similarity of raw ArcFace embeddings on the 6,000 official LFW pairs. This is a **sanity check** that the detection + alignment + embedding pipeline is correct. Later experiments compare this raw baseline against GRP and IoM protected templates.

**Needs:** `research/data/lfw_embeddings.npz` from `extract_lfw_embeddings.py` (full run). **Target:** 10-fold accuracy >= 99.5%.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import auc, roc_curve

DATA = Path("../data/lfw_embeddings.npz")  # notebooks run from research/notebooks
assert DATA.exists(), f"Run extract_lfw_embeddings.py first: {DATA.resolve()}"
d = np.load(DATA)
emb = d["emb"].astype(np.float64)
emb /= np.linalg.norm(emb, axis=1, keepdims=True)
pair_idx, issame, folds, det_failed = d["pair_idx"], d["issame"], d["folds"], d["det_failed"]
print(
    f"{len(issame)} pairs ({issame.sum()} genuine, {(~issame).sum()} impostor), "
    f"{len(emb)} images, {det_failed.sum()} detection failures"
)

## 1. Similarity score for every pair

In [ ]:
scores = np.einsum("ij,ij->i", emb[pair_idx[:, 0]], emb[pair_idx[:, 1]])  # cosine
print(f"genuine  mean {scores[issame].mean():.3f}   impostor mean {scores[~issame].mean():.3f}")

## 2. 10-fold accuracy

For each fold, the threshold is chosen on the other 9 folds and tested on the held-out fold, the standard LFW protocol.

In [ ]:
def best_threshold(s, y):
    """Threshold (predict same if score >= t) with the highest accuracy on (s, y)."""
    order = np.argsort(s)
    s, y = s[order], y[order]
    neg_before = np.concatenate([[0], np.cumsum(~y)])
    pos_before = np.concatenate([[0], np.cumsum(y)])
    correct = neg_before + (y.sum() - pos_before)
    i = int(np.argmax(correct))
    return s[i] if i < len(s) else s[-1] + 1e-6


accs, thresholds = [], []
for k in np.unique(folds):
    test = folds == k
    t = best_threshold(scores[~test], issame[~test])
    accs.append(((scores[test] >= t) == issame[test]).mean())
    thresholds.append(t)
acc_mean, acc_std = float(np.mean(accs)), float(np.std(accs))
print("per-fold accuracy:", np.round(accs, 4))
print(f"10-fold accuracy: {acc_mean * 100:.2f}% +- {acc_std * 100:.2f}%")
print(f"typical threshold: {np.mean(thresholds):.3f}")

## 3. ROC, AUC, EER and TAR at fixed FAR

In [ ]:
fpr, tpr, _ = roc_curve(issame, scores)
roc_auc = auc(fpr, tpr)
fnr = 1 - tpr
i = int(np.argmin(np.abs(fnr - fpr)))
eer = (fpr[i] + fnr[i]) / 2


def tar_at_far(far):
    return float(np.interp(far, fpr, tpr))


print(f"AUC            : {roc_auc:.5f}")
print(f"EER            : {eer * 100:.2f}%")
print(f"TAR @ FAR=1e-2 : {tar_at_far(1e-2) * 100:.2f}%")
print(f"TAR @ FAR=1e-3 : {tar_at_far(1e-3) * 100:.2f}%")
n_imp = int((~issame).sum())
print(
    f"\nNote: only {n_imp} impostor pairs, so FAR=1e-3 rests on about {n_imp // 1000} scores. "
    "The extended impostor set (later) makes this number trustworthy."
)

## 4. Score histograms and ROC curve

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
bins = np.linspace(-0.2, 1.0, 80)
ax1.hist(scores[~issame], bins=bins, alpha=0.7, label="impostor pairs", density=True)
ax1.hist(scores[issame], bins=bins, alpha=0.7, label="genuine pairs", density=True)
ax1.axvline(np.mean(thresholds), color="k", ls="--", label="threshold")
ax1.set(xlabel="cosine similarity", ylabel="density", title="Raw ArcFace scores on LFW")
ax1.legend()

m = fpr > 0
ax2.plot(fpr[m], tpr[m], label=f"raw (AUC={roc_auc:.4f})")
ax2.set_xscale("log")
ax2.set(xlabel="FAR (log)", ylabel="TAR", title="ROC", ylim=(0.9, 1.001))
ax2.grid(alpha=0.3)
ax2.legend()
plt.tight_layout()
plt.show()

## 5. Check against the target

In [ ]:
print(f"E1 raw baseline: {acc_mean * 100:.2f}% (target >= 99.50%)")
print("PASS" if acc_mean >= 0.995 else "CHECK: below target, see the notes below")
if acc_mean < 0.995:
    print("- Did you use --limit? A small subset gives noisy results; run the full extraction.")
    print(f"- Detection failures: {det_failed.sum()} (should be a handful at most).")
    print("- Embeddings must come from buffalo_l on uint8 BGR images (see the extract script).")